# 09 Run All Research Pipeline

Master runner notebook. It only orchestrates scripts; implementation stays in `src/` and `scripts/`.

Run cells from top to bottom. Each long experiment is in its own cell so you can resume from the next unfinished stage.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

def find_project_dir():
    candidates = [Path.cwd(), Path.cwd().parent]
    input_root = Path('/kaggle/input')
    if input_root.exists():
        candidates.extend(path.parent for path in input_root.rglob('configs/config.yaml'))
    for candidate in candidates:
        if (candidate / 'configs/config.yaml').exists() and (candidate / 'scripts').exists():
            return candidate.resolve()
    raise FileNotFoundError('Could not find project root with configs/config.yaml and scripts/.')

is_kaggle = Path('/kaggle/working').exists()
project_dir = find_project_dir()
if is_kaggle and str(project_dir).startswith('/kaggle/input'):
    work_project = Path('/kaggle/working/stanceeval_project')
    if work_project.exists():
        shutil.rmtree(work_project)
    shutil.copytree(project_dir, work_project)
    project_dir = work_project

os.chdir(project_dir)
os.environ.setdefault('STANCEEVAL_DATA_DIR', str(project_dir))
os.environ.setdefault('STANCEEVAL_OUTPUT_DIR', '/kaggle/working/outputs' if is_kaggle else 'outputs')
os.environ.setdefault('STANCEEVAL_RESULTS_DIR', '/kaggle/working/results' if is_kaggle else 'results')
os.environ.setdefault('STANCEEVAL_BEST_MODEL_DIR', '/kaggle/working/outputs/best_model' if is_kaggle else 'outputs/best_model')

print('Project:', project_dir)
print('Data:', os.environ['STANCEEVAL_DATA_DIR'])
print('Outputs:', os.environ['STANCEEVAL_OUTPUT_DIR'])
print('Results:', os.environ['STANCEEVAL_RESULTS_DIR'])

def run_stage(name, args):
    print('\n' + '=' * 80)
    print(name)
    print('=' * 80)
    command = [sys.executable, *args]
    print('Command:', ' '.join(command))
    subprocess.check_call(command)


In [ ]:
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])


## 00 Environment Check

In [ ]:
run_stage('00 Environment Check', [
    'scripts/smoke_test.py',
    '--config', 'configs/config.yaml',
    '--base-model-id', 'aubmindlab/bert-base-arabertv02-twitter',
    '--verbose',
])


## 01 Base Model Comparison

In [ ]:
run_stage('01 Base Model Comparison', [
    'scripts/compare_base_models.py',
    '--config', 'configs/config.yaml',
    '--verbose',
])


## 02 Layer-wise CLS Analysis

In [ ]:
run_stage('02 Layer-wise CLS Analysis', [
    'scripts/run_layerwise_cls_analysis.py',
    '--config', 'configs/config.yaml',
    '--verbose',
])


## 03 CLS4 Equal Loss

In [ ]:
run_stage('03 CLS4 Equal Loss', [
    'scripts/run_selected_layers_experiment.py',
    '--config', 'configs/config.yaml',
    '--experiment', 'cls4_equal_loss',
    '--verbose',
])


## 04 CLS4 Frozen Encoder

In [ ]:
run_stage('04 CLS4 Frozen Encoder', [
    'scripts/run_selected_layers_experiment.py',
    '--config', 'configs/config.yaml',
    '--experiment', 'cls4_frozen',
    '--verbose',
])


## 05 Weighted Logit Fusion

In [ ]:
run_stage('05 Weighted Logit Fusion', [
    'scripts/run_selected_layers_experiment.py',
    '--config', 'configs/config.yaml',
    '--experiment', 'weighted_logits',
    '--verbose',
])


## 06 Learnable Loss Weights

In [ ]:
run_stage('06 Learnable Loss Weights', [
    'scripts/run_selected_layers_experiment.py',
    '--config', 'configs/config.yaml',
    '--experiment', 'learnable_loss',
    '--verbose',
])


## 07 Final Analysis and Submission Generation

In [ ]:
run_stage('07 Final Analysis', [
    'scripts/final_analysis.py',
    '--config', 'configs/config.yaml',
    '--verbose',
])

run_stage('07 Submission Generation', [
    'scripts/generate_submissions.py',
    '--config', 'configs/config.yaml',
    '--verbose',
])


## 08 Show Final Outputs

In [ ]:
import json
import pandas as pd

results_dir = Path(os.environ['STANCEEVAL_RESULTS_DIR'])
comparison_path = results_dir / 'comparison.csv'
best_path = results_dir / 'best_experiment.json'

if comparison_path.exists():
    display(pd.read_csv(comparison_path).sort_values('dev_Favg2', ascending=False))
else:
    print('Missing:', comparison_path)

if best_path.exists():
    print(json.dumps(json.loads(best_path.read_text(encoding='utf-8')), indent=2, ensure_ascii=False))
else:
    print('Missing:', best_path)
